# The Four Pillars of OOP

In [1]:
import sys
from pathlib import Path

current = Path.cwd()
for parent in [current, *current.parents]:
    if (parent / '_config.yml').exists():
        project_root = parent
        break
else:
    project_root = Path.cwd().parent.parent

sys.path.insert(0, str(project_root))

from shared import thinkpython, diagram, jupyturtle, download

sys.modules['thinkpython'] = thinkpython
sys.modules['diagram'] = diagram
sys.modules['jupyturtle'] = jupyturtle
sys.modules['download'] = download


Object-oriented programming is often organized around four big ideas known as the **four pillars of OOP**. These pillars are not strict rules that every class must follow in the same way. Instead, they are design principles that help us build programs that are easier to understand, extend, and maintain.

When people say a program is "object-oriented," they usually mean that classes and objects are used to organize related data together with the operations that act on that data. The four pillars give us a vocabulary for describing how that organization works.

As you read each section, keep three questions in mind:

- What problem does this pillar solve?
- What does the Python code gain by using it?
- What would the code look like without it?

| Pillar | Core idea |
|--------|-----------|
| **Encapsulation** | Bundle data and behavior together, and protect internal state |
| **Inheritance** | Build a new class by reusing and extending an existing one |
| **Polymorphism** | Use a common interface even when different classes behave differently |
| **Abstraction** | Show what an object can do without exposing every implementation detail |

In practice, these ideas overlap. A class might use encapsulation to protect its data, inheritance to reuse behavior, polymorphism to work with many object types, and abstraction to present a simple public interface. We study them separately here so each one is easier to see.

Encapsulation comes first because it is one of the most immediate and practical ideas in everyday Python class design.

## Encapsulation

**Encapsulation** means that an object manages its own state (data at the point of time) by keeping related data and behavior together and by controlling how that data is accessed or changed.

This idea has two parts:

- A class bundles attributes and methods into one coherent unit.
- A class protects its internal state so values are changed in deliberate, valid ways rather than arbitrarily from outside the object.

That protection matters for at least two reasons:

- **Abstraction**: the class can expose a simple public interface while hiding implementation details.
- **Data integrity**: the class can validate updates and keep the object in a consistent state.

In Python, encapsulation is usually based on conventions and interface design rather than strict access control. A single leading underscore such as `_balance` means "internal use by convention." A double leading underscore such as `__salary` triggers **name mangling**, which makes accidental external access less likely. When we want controlled read or write access, `@property` gives us a clean public interface.

The main goal is to make callers depend on the object's **public interface**, not on the details of how the object stores or computes its data.

Consider a payroll system. An employee's name is usually safe to expose directly, but salary is more sensitive and should be accessed through a controlled interface. The example below uses a double-underscore attribute for the stored salary and a read-only `@property` for safe access.

In [2]:
class Employee:
    def __init__(self, name, salary):
        if salary < 0:
            raise ValueError('Salary cannot be negative')

        self.name = name            ### public attribute
        self.__salary = salary      ### double underscore triggers name mangling; this is meant
                                    ### for internal use and should not be accessed directly
                                    ### from outside the class.

    @property   ### This property lets us expose salary through a controlled read-only interface.
    def salary(self):
        return self.__salary

emp = Employee('Frederick', 50000)
print(emp.name)
print(emp.salary)   ### this calls the salary method, but we access it like an attribute
                    ### because of the @property decorator.

Frederick
50000


The same encapsulation idea appears in banking systems. A bank account should not allow outside code to change the balance arbitrarily; instead, the class should expose controlled operations such as `deposit` and `withdraw`, along with a read-only way to inspect the current balance.

In [3]:
class BankAccount:
    def __init__(self, owner, balance=0):
        if balance < 0:
            raise ValueError('Starting balance cannot be negative')

        self.owner = owner
        self._balance = balance     ### single underscore is a convention indicating that
                                    ### this attribute is intended for internal use, but it
                                    ### can still be accessed from outside the class.

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError('Deposit must be positive')
        self._balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError('Withdrawal must be positive')
        if amount > self._balance:
            raise ValueError('Insufficient funds')
        self._balance -= amount

acct = BankAccount('Alice', 100)
acct.deposit(50)
acct.withdraw(30)
print(acct.balance)  # 120

120


## Polymorphism

**Polymorphism** means different object types can respond to the same method call in their own way.

In the banking example below, each account type implements the same method, `month_end()`.
The caller does not need `if/elif` logic for each account type — it can just call one method name on all accounts.

In [4]:
class CheckingAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

    def month_end(self):
        self.balance -= 10   # monthly service fee


class SavingsAccount:
    def __init__(self, owner, balance, rate=0.01):
        self.owner = owner
        self.balance = balance
        self.rate = rate

    def month_end(self):
        self.balance += self.balance * self.rate   # monthly interest


class LoanAccount:
    def __init__(self, owner, balance, rate=0.015):
        self.owner = owner
        self.balance = balance
        self.rate = rate

    def month_end(self):
        self.balance += self.balance * self.rate   # interest on amount owed


accounts = [
    CheckingAccount('Alice', 1200),
    SavingsAccount('Bob', 5000),
    LoanAccount('Charlie', 2000),
]

In [5]:
print('Before month-end:')
for account in accounts:
    print(f'{account.owner}: {account.balance:.2f}')

Before month-end:
Alice: 1200.00
Bob: 5000.00
Charlie: 2000.00


All objects in `accounts` provide `month_end()`, so we can apply month-end updates with one uniform loop.
No type-specific `if/elif` branches are needed.

In [6]:
for account in accounts:
    account.month_end()   # polymorphic call

print('\nAfter month-end:')
for account in accounts:
    print(f'{account.owner}: {account.balance:.2f}')


After month-end:
Alice: 1190.00
Bob: 5050.00
Charlie: 2030.00


In this loop, `account` can be a `CheckingAccount`, `SavingsAccount`, or `LoanAccount`,
but the caller code is identical:

`account.month_end()`

Python dispatches to the correct class-specific implementation at runtime.
That is the core idea of **polymorphism**: one interface, many behaviors.

### Access Level Conventions

Python does not enforce strict access modifiers like Java or C++. Instead, it uses naming patterns to communicate intent:

- `name`: public
- `_name`: internal use by convention
- `__name`: triggers name mangling to reduce accidental access or accidental override in subclasses

That last point matters: double underscore is more than a social convention. Python actually rewrites the attribute name internally. It is still not true private security, but it does provide stronger protection against accidental misuse than a single underscore.

In the examples above:

- `Employee.__salary` shows double-underscore name mangling.
- `BankAccount._balance` shows an internal attribute exposed safely through methods and `@property`.

See the note below for how name mangling works.

:::{admonition} Name Mangling in Python
`__attribute` (double underscore) triggers **name mangling**, where Python rewrites the name internally to `_ClassName__attribute`.

This is designed to prevent accidental access or accidental overrides in subclasses, not to provide true security.

Example: `self.__salary` inside `Employee` is stored as `_Employee__salary`.

As for single leading underscore `_name`, it is a convention meaning "this is internal / don't touch it from outside the class." Python does nothing special: it's just a social contract between developers. The attribute is fully accessible, just signaled as private by convention.
:::

## Inheritance

The language feature most often associated with object-oriented programming is **inheritance**. Inheritance lets us define a new class as a modified or specialized version of an existing class.

When inheritance is used well, it helps us avoid rewriting code that is already correct in the parent class. The child class reuses what it needs and adds or overrides only the parts that make it different.

The examples in this section use products, inventories, and orders. That domain is closer to the kinds of systems used in business settings, but the main idea is general: a child class can reuse behavior from a parent class and then extend it for a more specific purpose.

In [7]:
# Setup: build a richer banking example for the Inheritance section.

class BankAccount:
    """Represents a bank account with an owner, balance, and account number."""

    def __init__(self, account_number, owner, balance=0.0):
        self.account_number = account_number
        self.owner = owner
        self.balance = balance

    def __str__(self):
        return f'{self.account_number}: {self.owner} - ${self.balance:.2f}'

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError('Deposit must be positive')
        self.balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError('Withdrawal must be positive')
        if amount > self.balance:
            raise ValueError('Insufficient funds')
        self.balance -= amount


### This base class will be extended by more specialized account types below.

The cells below use the `%%add_method_to` cell magic provided by the `thinkpython` module. It adds one method at a time to an existing class, so a class can be built incrementally across multiple cells instead of being written all at once. A cell that begins with `%%add_method_to SavingsAccount`, for example, adds the method it contains directly to the `SavingsAccount` class.

## Parents and children

Inheritance is the ability to define a new class that is a modified version of an existing class.
A useful banking example is a `SavingsAccount`, which is a specific kind of `BankAccount`.

* A savings account is similar to a general bank account because it still has an owner, an account number, a balance, and common operations such as deposit and withdrawal.

* A savings account is also different because it has behavior that does not apply to every account type. For example, it may earn interest at the end of each month.

This relationship between classes, where one is a specialized version of another, lends itself naturally to inheritance.

To define a new class that is based on an existing class, the name of the existing class is placed in parentheses.

In [8]:
class SavingsAccount(BankAccount):
    """Represents a savings account."""

This definition indicates that `SavingsAccount` inherits from `BankAccount`, which means that `SavingsAccount` objects can access methods defined in `BankAccount`, like `deposit` and `withdraw`.

`SavingsAccount` also inherits `__init__` from `BankAccount`, but if we define `__init__` in the `SavingsAccount` class, it overrides the one in the `BankAccount` class.

In [9]:
%%add_method_to SavingsAccount

    def __init__(self, account_number, owner, balance=0.0, rate=0.01):
        BankAccount.__init__(self, account_number, owner, balance)
        self.rate = rate    ### extra attribute specific to SavingsAccount

This version of `__init__` takes the same basic account information as the parent class and adds an interest rate.
When a `SavingsAccount` is created, Python invokes this method, but `super().__init__(...)` reuses the parent setup so the shared attributes do not have to be written again.

In [10]:
savings = SavingsAccount('S-1001', 'Ava', 1200.00, rate=0.02)
print(savings.rate)

0.02


Because `SavingsAccount` inherits from `BankAccount`, it can use methods defined in the parent class without redefining them. The next example calls `deposit`, which was written once in `BankAccount` and then reused by the child class.

In [11]:
savings.deposit(300.00)
print(savings)
print(isinstance(savings, SavingsAccount))
print(isinstance(savings, BankAccount))

S-1001: Ava - $1500.00
True
True


The next example adds a method that is shared by every kind of bank account. A transfer operation belongs in the parent class because it is useful for a regular bank account, a savings account, and any future subclass that behaves like a bank account.

In [12]:
%%add_method_to BankAccount

    def transfer_to(self, other_account, amount):
        if not isinstance(other_account, BankAccount):
            raise TypeError('Transfers require another BankAccount')
        self.withdraw(amount)
        other_account.deposit(amount)

This method is defined once in the parent class, but it can be used by any child class that inherits from `BankAccount`. That is one of the main advantages of inheritance: common behavior lives in one place instead of being duplicated across several related classes.

A child class is often called a *subclass*, and the parent class is often called a *superclass*. In this example, `SavingsAccount` is a subclass of `BankAccount`, and `BankAccount` is a superclass of `SavingsAccount`.

A `SavingsAccount` should be usable anywhere a `BankAccount` is expected because it still supports the same core account behaviors. That idea makes inheritance practical: the parent class defines what related objects have in common, and the child class adds what makes it special.

## Specialization

Inheritance becomes especially useful when subclasses need to keep the basic behavior of the parent class but also add rules of their own.
A `BusinessAccount`, for example, is still a bank account, but it may charge a service fee for certain withdrawals. That makes it a good example of specialization.

In [13]:
class BusinessAccount(BankAccount):
    """Represents a business checking account."""

    def __init__(self, account_number, owner, balance=0.0, transaction_fee=5.0):
        BankAccount.__init__(self, account_number, owner, balance)
        self.transaction_fee = transaction_fee

Suppose the account charges a service fee every time a special withdrawal is made. Given the withdrawal amount, we can compute the total cost of the transaction by adding the fee.

In [14]:
amount = 200
fee = 7.50
amount + fee

207.5

The following method performs that withdrawal and applies the transaction fee. It belongs in the child class because the extra fee is specific to business accounts, not to every kind of bank account.

In [15]:
%%add_method_to BusinessAccount

    def withdraw_with_fee(self, amount):
        total = amount + self.transaction_fee
        self.withdraw(total)
        return total

In [16]:
business = BusinessAccount('B-2001', 'Northwind Supply', 5000.00, transaction_fee=12.50)
personal = SavingsAccount('S-2002', 'Mina', 1800.00, rate=0.03)

business.transfer_to(personal, 400.00)
print(business)
print(personal)

B-2001: Northwind Supply - $4600.00
S-2002: Mina - $2200.00


The `transfer_to` method came from the parent class, but it still works with a `BusinessAccount` and a `SavingsAccount`. That is what makes the inheritance relationship useful: the child classes gain the shared behavior automatically.

In [17]:
charged = business.withdraw_with_fee(250.00)
print(f'Total charged: ${charged:.2f}')
print(business)

Total charged: $262.50
B-2001: Northwind Supply - $4337.50


The new method subtracts both the withdrawal amount and the fee, while the inherited methods from `BankAccount` still handle the core balance updates.

In [18]:
business.balance

4337.5

In this example, `BusinessAccount` inherited general account behavior from `BankAccount` and then added specialized behavior of its own. The child class did not need to rewrite deposit, basic withdrawal, string display, or transfer logic.

The `SavingsAccount` and `BusinessAccount` classes inherited all the methods from `BankAccount`, such as `deposit`, `withdraw`, `__str__`, and `transfer_to`, but each subclass also gained attributes or methods that reflect its own role in the banking system.

In [19]:
### Exercise: Banking Polymorphism
#   1. Define a class `InvestmentAccount` with `owner`, `balance`, and `rate`.
#   2. Add a `month_end(self)` method that applies monthly growth:
#         self.balance += self.balance * self.rate
#   3. Create an `InvestmentAccount('Diana', 3000, 0.02)`.
#   4. Put it in a list with existing `accounts`, call `month_end()` on each object,
#      and print each owner's updated balance.
### Your code starts here.



### Your code ends here.

In [20]:
### Solution
class InvestmentAccount:
    def __init__(self, owner, balance, rate=0.02):
        self.owner = owner
        self.balance = balance
        self.rate = rate

    def month_end(self):
        self.balance += self.balance * self.rate

### Rebuild fresh objects from the original polymorphism examples.
### This avoids reusing already-mutated objects and avoids later class-name collisions.
checking_type = type(accounts[0])
savings_type = type(accounts[1])
loan_type = type(accounts[2])

base_accounts = [
    checking_type('Alice', 1200),
    savings_type('Bob', 5000),
    loan_type('Charlie', 2000),
]
portfolio = base_accounts + [InvestmentAccount('Diana', 3000, 0.02)]

for account in portfolio:
    account.month_end()   # polymorphic call

for account in portfolio:
    print(f'{account.owner}: {account.balance:.2f}')

Alice: 1190.00
Bob: 5050.00
Charlie: 2030.00
Diana: 3060.00


## `super()` and `issubclass()`

These two tools appear often in inheritance, but they solve different problems.

- `super()` helps a child class reuse behavior from its parent, especially during initialization.
- `issubclass()` checks whether one class inherits from another at runtime.

### `super()`

`super()` returns a proxy object that makes it possible to call a method from a parent class without naming that parent explicitly. This is especially useful in `__init__`, where the child class often relies on the parent class to set up shared state before adding child-specific attributes.

In [24]:
class Animal:
    def __init__(self, name, sound):
        self.name = name
        self.sound = sound

    def speak(self):
        return f'{self.name} says {self.sound}!'

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name, sound='Woof')  # call parent __init__ first
        self.breed = breed                    # then add child-specific data

    def info(self):
        return f'{self.name} is a {self.breed}'

d = Dog('Rex', 'Labrador')
print(d.speak())   # inherited behavior uses parent setup
print(d.info())    # child-specific behavior

Rex says Woof!
Rex is a Labrador


### `issubclass()`

`issubclass(Child, Parent)` returns `True` if `Child` is a subclass of `Parent` or is `Parent` itself. It is useful when class relationships need to be checked at runtime.

A practical detail matters here: the first argument to `issubclass()` must itself be a class. If user input or function arguments are being validated, that point is often worth checking explicitly before calling `issubclass()`.

| Function | Checks |
|---|---|
| `isinstance(obj, cls)` | Is `obj` an instance of `cls` or one of its subclasses? |
| `issubclass(A, B)` | Is class `A` a subclass of `B`? |


In [25]:
print(issubclass(Dog, Animal))   # True
print(issubclass(Animal, Dog))   # False
print(issubclass(Dog, Dog))      # True (a class is a subclass of itself)

# Common use: guarding against wrong arguments
def make_animal(cls, *args, **kwargs):
    if not isinstance(cls, type):
        raise TypeError('cls must be a class')
    if not issubclass(cls, Animal):
        raise TypeError(f'{cls} is not an Animal subclass')
    return cls(*args, **kwargs)

rex = make_animal(Dog, 'Rex', 'Labrador')
print(rex.speak())   # Rex says Woof!

True
False
True
Rex says Woof!


## Abstraction

**Abstraction** means defining a clean interface while hiding lower-level implementation details. The user of an object should understand what the object can do without needing to know exactly how the object does it.

Python's `abc` module provides **Abstract Base Classes**. An abstract base class defines the methods that subclasses are required to provide, but it does not force all subclasses to use the same internal implementation. In that sense, abstraction describes a contract: every concrete subclass must satisfy the same interface.

In [26]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    """Abstract base: every Shape must implement area() and perimeter()."""

    @abstractmethod
    def area(self): ...

    @abstractmethod
    def perimeter(self): ...

    def describe(self):
        return f'{type(self).__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}'


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

    def perimeter(self):
        return 2 * math.pi * self.radius


class Rectangle(Shape):
    def __init__(self, w, h):
        self.w = w
        self.h = h

    def area(self):
        return self.w * self.h

    def perimeter(self):
        return 2 * (self.w + self.h)


### Both objects satisfy the same abstract interface, so the loop is uniform.
for s in [Circle(5), Rectangle(4, 6)]:
    print(s.describe())
# Shape()  -> TypeError: cannot instantiate an abstract class directly

Circle: area=78.54, perimeter=31.42
Rectangle: area=24.00, perimeter=20.00


## Summary

The four pillars are easiest to understand separately, but good object-oriented design usually combines them:

- **Encapsulation** protects state and funnels access through a controlled interface.
- **Inheritance** lets a new class reuse and extend the behavior of an existing one.
- **Polymorphism** lets the same caller code work with different object types as long as they support the same interface.
- **Abstraction** defines what an object must provide without exposing every implementation detail.

When you read or design a class, a useful habit is to ask: What data does this object protect? What behavior does it inherit? What interface can be used polymorphically? What details are intentionally hidden?